In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score
import xgboost as xgb
from catboost import CatBoostClassifier
import mlflow
from pathlib import Path
import time

# ==========================================
# 1. SETUP PATH & LOAD DATA
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("07c_Epic_Trinity_Comparison")

print("🔍 1. Memuat Data Latih (Balanced) dan Uji (Murni)...")
train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")
test_df = pd.read_csv(root_path / "Data/split/test_data.csv")

# 35 Fitur inti hasil MFO
features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1', 'VCL4', 'VCL5', 'VCL6', 
                     'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'gender', 'engnat', 'age', 'religion', 'orientation',
                     'race', 'voted', 'married', 'familysize']
targets = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_train, Y_train = train_df[features], train_df[targets]
X_test, Y_test = test_df[features], test_df[targets]

# ==========================================
# 2. HYPERPARAMETER TUNING YANG ADIL
# ==========================================
print("\n⚖️ 2. Memulai Tuning Adil (Apple-to-Apple) untuk 3 Raksasa...")
print("Catatan: Tuning dilakukan pada 1 target utama (Depresi) agar komputasi efisien.")

# -- A. Tuning Random Forest --
print("   -> 🌲 Tuning Random Forest...")
rf_grid = {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, None]}
rf_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_grid, n_iter=5, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
rf_search.fit(X_train, Y_train['risk_depression'])
tuned_rf = rf_search.best_estimator_

# -- B. Tuning XGBoost --
print("   -> 🚀 Tuning XGBoost...")
xgb_grid = {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1]}
xgb_search = RandomizedSearchCV(xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42), xgb_grid, n_iter=5, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
xgb_search.fit(X_train, Y_train['risk_depression'])
tuned_xgb = xgb_search.best_estimator_

# -- C. Tuning CatBoost --
print("   -> 😼 Tuning CatBoost...")
cat_grid = {'iterations': [100, 200, 300], 'depth': [4, 6, 8], 'learning_rate': [0.01, 0.05, 0.1]}
# logging_level='Silent' agar terminal tidak banjir teks
cat_search = RandomizedSearchCV(CatBoostClassifier(logging_level='Silent', random_seed=42), cat_grid, n_iter=5, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
cat_search.fit(X_train, Y_train['risk_depression'])
tuned_cat = cat_search.best_estimator_

# ==========================================
# 3. PERTANDINGAN CLASSIFIER CHAIN
# ==========================================
print("\n🥊 3. Memasuki Arena Multi-label (Classifier Chain)...")

chains = {
    "Random Forest": ClassifierChain(tuned_rf, order='random', random_state=42),
    "XGBoost": ClassifierChain(tuned_xgb, order='random', random_state=42),
    "CatBoost": ClassifierChain(tuned_cat, order='random', random_state=42)
}

results = {}

with mlflow.start_run(run_name="Epic_Trinity_Comparison"):
    for name, model in chains.items():
        print(f"   ⏳ Melatih dan Menguji {name}...")
        start_time = time.time()
        
        # Latih model
        model.fit(X_train, Y_train)
        
        # Prediksi Data Test
        Y_pred = model.predict(X_test)
        
        # Evaluasi
        macro_f1 = f1_score(Y_test, Y_pred, average='macro')
        execution_time = time.time() - start_time
        
        results[name] = {"Macro F1": macro_f1, "Time (s)": execution_time}
        mlflow.log_metric(f"{name.replace(' ', '_')}_Macro_F1", macro_f1)

    # ==========================================
    # 4. PAPAN KLASEMEN (LEADERBOARD)
    # ==========================================
    print("\n" + "🌟"*25)
    print("🏆 KLASEMEN AKHIR: BATTLE ROYALE ALGORITMA 🏆")
    print("🌟"*25)
    
    # Mengurutkan berdasarkan F1-Score tertinggi
    sorted_results = sorted(results.items(), key=lambda x: x[1]['Macro F1'], reverse=True)
    
    for rank, (name, metrics) in enumerate(sorted_results, 1):
        print(f"{rank}. {name.ljust(15)} | Macro F1: {metrics['Macro F1']:.4f} | Waktu Eksekusi: {metrics['Time (s)']:.2f} detik")
    
    print("\n✅ KESIMPULAN:")
    print(f"Pemenang mutlak di dataset ini adalah **{sorted_results[0][0]}**!")

d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


🔍 1. Memuat Data Latih (Balanced) dan Uji (Murni)...

⚖️ 2. Memulai Tuning Adil (Apple-to-Apple) untuk 3 Raksasa...
Catatan: Tuning dilakukan pada 1 target utama (Depresi) agar komputasi efisien.
   -> 🌲 Tuning Random Forest...
   -> 🚀 Tuning XGBoost...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [18:57:12] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


   -> 😼 Tuning CatBoost...

🥊 3. Memasuki Arena Multi-label (Classifier Chain)...
   ⏳ Melatih dan Menguji Random Forest...
   ⏳ Melatih dan Menguji XGBoost...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [18:57:40] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [18:57:41] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


   ⏳ Melatih dan Menguji CatBoost...

🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
🏆 KLASEMEN AKHIR: BATTLE ROYALE ALGORITMA 🏆
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
1. Random Forest   | Macro F1: 0.7876 | Waktu Eksekusi: 11.61 detik
2. XGBoost         | Macro F1: 0.7853 | Waktu Eksekusi: 2.12 detik
3. CatBoost        | Macro F1: 0.7836 | Waktu Eksekusi: 5.80 detik

✅ KESIMPULAN:
Pemenang mutlak di dataset ini adalah **Random Forest**!
